In [1]:
#The following file paths are all absolute paths. You can replace them with relative paths at runtime, and the files are located in their respective folders.
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import gym
import matplotlib.pyplot as plt
import random
import argparse
from collections import OrderedDict
from copy import copy
# import Learn_Knonlinear as lka
import scipy
import scipy.linalg
from scipy.integrate import odeint
import sys
import os
import csv
sys.path.append("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/Predict_Model_Train/")
sys.path.append("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/utility_LSPN/")
from Utility import data_collecter
from LSPN_test import LSPN_Mamba
os.environ['KMP_DUPLICATE_LIB_OK'] = "TRUE"


/home/serena/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
Methods = ["KoopmanDerivative","KoopmanRBF",\
            "KNonlinear","KNonlinearRNN","KoopmanU",\
            "KoopmanNonlinearA","KoopmanNonlinear",\
                "KNonlinearmamba"]
Method_names = ["KoopmanDerivative","KoopmanRBF",\
            "KDNN","KRNN","DKUC(ours)",\
            "DKAC(ours)","DKN(ours)",\
                "KNonlinearmamba"]

used for dampingpendulum, pendulum and mountaincar

In [ ]:
def short_predict(method_index,data,net,u_dim=1,Nstate=4):
    steps,train_traj_num,Nstates = data.shape
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = torch.FloatTensor(data).to(device)
    if method_index!=7:
        X_current = net.encode(data[0,:,u_dim:]).T
    max_loss_list = []
    mean_loss_list = []
    min_loss_list = []
    pre_list = []
    real_list = []
    if method_index==7:
        X_pred = net.forward(data[:steps-1,:,:].transpose(0, 1))
        X_pred = X_pred.transpose(0, 1)
        X_pred = X_pred[:,:,u_dim:]
    for i in range(steps-1):
        if method_index==4:
            X_current = net.forward(X_current.T,data[i,:,:u_dim]).T
        elif method_index==5:
            bilinear = net.bicode(X_current[:Nstate,:].T.detach(),data[i,:,:u_dim]).T #detach's problem 
            X_current = net.forward(X_current.T,bilinear.T).T
        elif method_index==6:
            bilinear = net.bicode(X_current[:Nstate,:].T.detach(),data[i,:,:u_dim]).T #detach's problem 
            X_current = net.forward(X_current.T,bilinear.T).T
        elif method_index==7:     
            X_current = X_pred[i,:,:]
        Y = data[i+1,:,u_dim:]
        if method_index==7:
            Err = X_current-Y
            print(torch.mean(X_current,axis=0)[0],torch.mean(Y,axis=0)[0])
            # print(Y.shape)
            pre_list.append(np.array(torch.mean(X_current,axis=0).detach().cpu().numpy()))
            # print(torch.mean(X_current[:Nstate,:].T,axis=0).shape)
            real_list.append(np.array(torch.mean(Y,axis=0).detach().cpu().numpy()))
        else:
            Err = X_current[:Nstate,:].T-Y
            print(torch.mean(X_current[:Nstate,:].T,axis=0)[0],torch.mean(Y,axis=0)[0])
            # print(Y.shape)
            pre_list.append(np.array(torch.mean(X_current[:Nstate,:].T,axis=0).detach().cpu().numpy()))
            # print(torch.mean(X_current[:Nstate,:].T,axis=0).shape)
            real_list.append(np.array(torch.mean(Y,axis=0).detach().cpu().numpy()))
        max_loss_list.append(torch.mean(torch.max(torch.abs(Err),axis=0).values).detach().cpu().numpy())
        mean_loss_list.append(torch.mean(torch.mean(torch.abs(Err),axis=0)).detach().cpu().numpy())
        min_loss_list.append(torch.mean(torch.min(torch.abs(Err),axis=0).values).detach().cpu().numpy())
    print(np.array(pre_list)[:,0], np.array(real_list)[:,0])
    print(np.array(mean_loss_list))
    return np.array(max_loss_list),np.array(mean_loss_list),np.array(min_loss_list),np.array(pre_list),np.array(real_list)

In [ ]:
def mean_predict(suffix,env_name,method_index,layer_i,steps):
    # method_index = 0
    method = Methods[method_index]
    root_path = "/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/Mamba_data_raw/"+suffix
    print(method)
    #sys.path.append("control/train/")
    if  method.endswith("KNonlinearRNN"):
        import Learn_Knonlinear_RNN as lka
    elif method.endswith("KNonlinear"):
        import Learn_Knonlinear as lka
    elif method.endswith("KoopmanNonlinear"):
        import learn_DKN_SOC_sizeNN as lka
    elif method.endswith("KoopmanNonlinearA"):
        import learn_DKAC_SOC_sizeNN as lka
    elif method.endswith("KoopmanU"):
        import learn_DKUC_SOC_sizeNN as lka
    elif method.endswith("KNonlinearmamba"):
        import Learn_Knonlinear_mamba as lka  
    for file in os.listdir(root_path):
        if file.startswith(method+"_"+env_name) and file.endswith(".pth"):
            model_path = file  
    Data_collect = data_collecter(env_name)
    udim = Data_collect.udim
    Nstates = Data_collect.Nstates
    layer_depth = layer_i
    layer_width = 128
    dicts = torch.load(root_path+"/"+model_path,map_location=torch.device('cpu'))
    state_dict = dicts["model"]
    if method.endswith("KNonlinear"):
        Elayer = dicts["Elayer"]
        net = lka.Network(layers=Elayer,u_dim=udim)
    elif method.endswith("KNonlinearRNN"):
        net = lka.Network(input_size=udim+Nstates,output_size=Nstates,hidden_dim=layer_width, n_layers=layer_depth-1)
    elif method.endswith("KNonlinearmamba"):
        net = LSPN_Mamba(
        # This module uses roughly 3 * expand * d_model^2 parameters
        d_model=3, # Model dimension d_model
        d_state=8,  # SSM state expansion factor
        d_conv=4,    # Local convolution width
        expand=4,    # Block expansion factor
    ).to("cuda")
    elif method.endswith("KoopmanNonlinear") or method.endswith("KoopmanNonlinearA"):
        layer = dicts["layer"]
        blayer = dicts["blayer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,blayer,NKoopman,udim)
    elif method.endswith("KoopmanU"):
        layer = dicts["layer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,NKoopman,udim)  
    net.load_state_dict(state_dict)
    #device = torch.device("cpu")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.cuda()
    net.double()
    Samples = 1
    steps = steps
    random.seed(2022)
    np.random.seed(2022)
    times = 100
    pre_list_all = np.zeros((times,steps,Nstates))
    rea_list_all = np.zeros((times,steps,Nstates))
    with torch.no_grad():
        for i in range(times):
            # test_data_path = "D:/毕业设计/中期/Python/MPC_trykoopman/results/SOC_compare_sizeNNdata/"+"method{}{}.npy".format(env_name,i)
            # if os.path.exists(test_data_path):
            #     test_data = np.load("D:/毕业设计/中期/Python/MPC_trykoopman/results/SOC_compare_sizeNNdata/{}{}.npy".format(env_name,i))
            # else:
            test_data = Data_collect.collect_koopman_data(Samples,steps)
            np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_data/middle_data/heatdatapredict_future{}{}{}.npy".format(i,suffix,env_name),test_data)
            _,_,_,pre_list,rea_list = short_predict(method_index,test_data,net,udim,Nstate=Nstates)
            #print(pre_list.shape)
            pre_list_all[i,:,:] = pre_list
            rea_list_all[i,:,:] = rea_list
    pre_list_mean = pre_list_all#np.mean(pre_list_all,axis=0)
    rea_list_mean = rea_list_all#np.mean(rea_list_all,axis=0)  
    # np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/"+env_name+"_"+method+"layer1{}{}.npy".format(layer_i, steps),np.array([pre_list_mean, rea_list_mean]))
    np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/heatdata"+env_name+"{}-pre-long{}.npy".format(method_index,steps),pre_list_mean)
    np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/heatdata"+env_name+"{}-real-long{}.npy".format(method_index,steps),rea_list_mean)
    return pre_list_mean, rea_list_mean

In [ ]:
#suffix = "compare_DKAC_sizeNN_30"
#suffix = "DKUC_SOC_sizeNN"
#suffix = "DKN_SOC_sizeNN"
suffix = ["*","*","Knolinear_SOC_models","KRNN_SOC_models","DKUC_SOC_sizeNN","compare_DKAC_sizeNN_30","DKN_SOC_sizeNN","mamba_test_50_5k_2w"]
env_name = "DampingPendulum"
env_name = "MountainCarContinuous-v0"
# env_name = "CartPole-v1"
# env_name = "Pendulum-v1"
steps = 100
for i in range(7,8):
    pre_list_mean, rea_list_mean = mean_predict(suffix[i],env_name,method_index=i,layer_i=2,steps=steps)
    print(pre_list_mean.shape)

used for luorenz and henon_map

In [3]:
def short_predict(method_index,data,net):
    train_traj_num,steps,Nstates = data.shape
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = torch.FloatTensor(data).to(device)
    max_loss_list = []
    mean_loss_list = []
    min_loss_list = []
    pre_list = []
    real_list = []
    if method_index==7:
        X_pred = net.forward(data[:,:steps-1,:])
        # X_pred = X_pred
        # X_pred = X_pred[:,:,u_dim:]
    for i in range(steps-1):
        if method_index==7:     
            X_current = X_pred[:,i,:]
        Y = data[:,i+1,:]
        if method_index==7:
            Err = X_current-Y
            print(torch.mean(X_current,axis=0)[0],torch.mean(Y,axis=0)[0])
            # print(Y.shape)
            pre_list.append(np.array(torch.mean(X_current,axis=0).detach().cpu().numpy()))
            # print(torch.mean(X_current[:Nstate,:].T,axis=0).shape)
            real_list.append(np.array(torch.mean(Y,axis=0).detach().cpu().numpy()))
        else:
            Err = X_current[:2,:].T-Y
            print(torch.mean(X_current[:2,:].T,axis=0)[0],torch.mean(Y,axis=0)[0])
            # print(Y.shape)
            pre_list.append(np.array(torch.mean(X_current[:2,:].T,axis=0).detach().cpu().numpy()))
            # print(torch.mean(X_current[:Nstate,:].T,axis=0).shape)
            real_list.append(np.array(torch.mean(Y,axis=0).detach().cpu().numpy()))
        max_loss_list.append(torch.mean(torch.max(torch.abs(Err),axis=0).values).detach().cpu().numpy())
        mean_loss_list.append(torch.mean(torch.mean(torch.abs(Err),axis=0)).detach().cpu().numpy())
        min_loss_list.append(torch.mean(torch.min(torch.abs(Err),axis=0).values).detach().cpu().numpy())
    print(np.array(pre_list)[:,0], np.array(real_list)[:,0])
    print(np.array(mean_loss_list))
    return np.array(max_loss_list),np.array(mean_loss_list),np.array(min_loss_list),np.array(pre_list),np.array(real_list)

In [6]:
def read_lorenz_dataset_original_shape(file_path, num_samples=100000, num_steps=100):
    data = np.zeros((num_samples, num_steps, 3))
    with open(file_path, 'r', newline='') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)  # 跳过列名行
        for i in range(num_samples):
            for j in range(num_steps):
                row = next(reader)
                data[i, j] = [float(val) for val in row]
    return data

def mean_predict(suffix,env_name,method_index,layer_i,steps):
    # method_index = 0
    method = Methods[method_index]
    root_path = "/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/Mamba_data_raw/"+suffix
    print(method)
    #sys.path.append("control/train/")
    if  method.endswith("KNonlinearRNN"):
        import Learn_Knonlinear_RNN as lka
    elif method.endswith("KNonlinear"):
        import Learn_Knonlinear as lka
    elif method.endswith("KoopmanNonlinear"):
        import learn_DKN_SOC_sizeNN as lka
    elif method.endswith("KoopmanNonlinearA"):
        import learn_DKAC_SOC_sizeNN as lka
    elif method.endswith("KoopmanU"):
        import learn_DKUC_SOC_sizeNN as lka
    elif method.endswith("KNonlinearmamba"):
        import Learn_Knonlinear_mamba as lka  
    for file in os.listdir(root_path):
        if file.startswith(method+"_"+"luorenz") and file.endswith(".pth"):
            model_path = file  
    # Data_collect = data_collecter(env_name)
    udim = 1
    Nstates = 2
    layer_depth = layer_i
    layer_width = 128
    dicts = torch.load(root_path+"/"+model_path,map_location=torch.device('cpu'))
    state_dict = dicts["model"]
    if method.endswith("KNonlinear"):
        Elayer = dicts["Elayer"]
        net = lka.Network(layers=Elayer,u_dim=0)
    elif method.endswith("KNonlinearRNN"):
        net = lka.Network(input_size=0+2,output_size=Nstates,hidden_dim=layer_width, n_layers=layer_depth-1)
    elif method.endswith("KNonlinearmamba"):
        net = LSPN_Mamba(
        # This module uses roughly 3 * expand * d_model^2 parameters
        d_model=3, # Model dimension d_model
        d_state=8,  # SSM state expansion factor
        d_conv=4,    # Local convolution width
        expand=4,    # Block expansion factor
    ).to("cuda")
    elif method.endswith("KoopmanNonlinear") or method.endswith("KoopmanNonlinearA"):
        layer = dicts["layer"]
        blayer = dicts["blayer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,blayer,NKoopman,udim)
    elif method.endswith("KoopmanU"):
        layer = dicts["layer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,NKoopman,udim)  
    net.load_state_dict(state_dict)
    #device = torch.device("cpu")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.cuda()
    net.double()
    Samples = 1
    steps = steps
    random.seed(2022)
    np.random.seed(2022)
    times = 100
    pre_list_all = np.zeros((times,steps-1,3))
    rea_list_all = np.zeros((times,steps-1,3))
    with torch.no_grad():
        for i in range(times):
            # test_data_path = "D:/毕业设计/中期/Python/MPC_trykoopman/results/SOC_compare_sizeNNdata/"+"method{}{}.npy".format(env_name,i)
            # if os.path.exists(test_data_path):
            #     test_data = np.load("D:/毕业设计/中期/Python/MPC_trykoopman/results/SOC_compare_sizeNNdata/{}{}.npy".format(env_name,i))
            # else:
            X_original_shape = read_lorenz_dataset_original_shape('/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/utility_LSPN/lorenz_data1000.csv',num_steps=steps)
            test_data = X_original_shape[-Samples:, :steps, :]
            np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_data/middle_data/heatdatapredict_future{}{}{}.npy".format(i,suffix,env_name),test_data)
            _,_,_,pre_list,rea_list = short_predict(method_index,test_data,net)
            #print(pre_list.shape)
            pre_list_all[i,:,:] = pre_list
            rea_list_all[i,:,:] = rea_list
    pre_list_mean = pre_list_all#np.mean(pre_list_all,axis=0)
    rea_list_mean = rea_list_all#np.mean(rea_list_all,axis=0)  
    # np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/"+env_name+"_"+method+"layer1{}{}.npy".format(layer_i, steps),np.array([pre_list_mean, rea_list_mean]))
    np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/heatdata"+env_name+"{}-pre-long{}.npy".format(method_index,steps),pre_list_mean)
    np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/heatdata"+env_name+"{}-real-long{}.npy".format(method_index,steps),rea_list_mean)
    return pre_list_mean, rea_list_mean

In [7]:
suffix = ["*","*","Knolinear_SOC_models","KRNN_SOC_models","DKUC_SOC_sizeNN","compare_DKAC_sizeNN_30","DKN_SOC_sizeNN","mamba_testluorenz2"]
env_name = "luorenz"

steps = 101
for i in range(7,8):
    pre_list_mean, rea_list_mean = mean_predict(suffix[i],env_name,method_index=i,layer_i=2,steps=steps)
    print(pre_list_mean.shape)

KNonlinearmamba
tensor(6.3376, device='cuda:0') tensor(6.3386, device='cuda:0')
tensor(5.9655, device='cuda:0') tensor(5.9539, device='cuda:0')
tensor(5.6159, device='cuda:0') tensor(5.5984, device='cuda:0')
tensor(5.2889, device='cuda:0') tensor(5.2732, device='cuda:0')
tensor(4.9936, device='cuda:0') tensor(4.9789, device='cuda:0')
tensor(4.7291, device='cuda:0') tensor(4.7157, device='cuda:0')
tensor(4.4950, device='cuda:0') tensor(4.4832, device='cuda:0')
tensor(4.2907, device='cuda:0') tensor(4.2802, device='cuda:0')
tensor(4.1148, device='cuda:0') tensor(4.1054, device='cuda:0')
tensor(3.9663, device='cuda:0') tensor(3.9579, device='cuda:0')
tensor(3.8441, device='cuda:0') tensor(3.8368, device='cuda:0')
tensor(3.7471, device='cuda:0') tensor(3.7410, device='cuda:0')
tensor(3.6744, device='cuda:0') tensor(3.6695, device='cuda:0')
tensor(3.6250, device='cuda:0') tensor(3.6211, device='cuda:0')
tensor(3.5978, device='cuda:0') tensor(3.5947, device='cuda:0')
tensor(3.5917, device='c

used for henon_map

In [3]:
def short_predict(method_index,data,net):
    train_traj_num,steps,Nstates = data.shape
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = torch.FloatTensor(data).to(device)
    max_loss_list = []
    mean_loss_list = []
    min_loss_list = []
    pre_list = []
    real_list = []
    if method_index==7:
        X_pred = net.forward(data[:,:steps-1,:])
        # X_pred = X_pred
        # X_pred = X_pred[:,:,u_dim:]
    for i in range(steps-1):
        if method_index==7:     
            X_current = X_pred[:,i,:]
        Y = data[:,i+1,:]
        if method_index==7:
            Err = X_current-Y
            print(torch.mean(X_current,axis=0)[0],torch.mean(Y,axis=0)[0])
            # print(Y.shape)
            pre_list.append(np.array(torch.mean(X_current,axis=0).detach().cpu().numpy()))
            # print(torch.mean(X_current[:Nstate,:].T,axis=0).shape)
            real_list.append(np.array(torch.mean(Y,axis=0).detach().cpu().numpy()))
        else:
            Err = X_current[:2,:].T-Y
            print(torch.mean(X_current[:2,:].T,axis=0)[0],torch.mean(Y,axis=0)[0])
            # print(Y.shape)
            pre_list.append(np.array(torch.mean(X_current[:2,:].T,axis=0).detach().cpu().numpy()))
            # print(torch.mean(X_current[:Nstate,:].T,axis=0).shape)
            real_list.append(np.array(torch.mean(Y,axis=0).detach().cpu().numpy()))
        max_loss_list.append(torch.mean(torch.max(torch.abs(Err),axis=0).values).detach().cpu().numpy())
        mean_loss_list.append(torch.mean(torch.mean(torch.abs(Err),axis=0)).detach().cpu().numpy())
        min_loss_list.append(torch.mean(torch.min(torch.abs(Err),axis=0).values).detach().cpu().numpy())
    print(np.array(pre_list)[:,0], np.array(real_list)[:,0])
    print(np.array(mean_loss_list))
    return np.array(max_loss_list),np.array(mean_loss_list),np.array(min_loss_list),np.array(pre_list),np.array(real_list)

In [4]:
def read_henon_map_dataset_original_shape(file_path):
    data = np.load(file_path)
    return data

def mean_predict(suffix,env_name,method_index,layer_i,steps):
    # method_index = 0
    method = Methods[method_index]
    root_path = "/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/Mamba_data_raw/"+suffix
    print(method)
    #sys.path.append("control/train/")
    if  method.endswith("KNonlinearRNN"):
        import Learn_Knonlinear_RNN as lka
    elif method.endswith("KNonlinear"):
        import Learn_Knonlinear as lka
    elif method.endswith("KoopmanNonlinear"):
        import learn_DKN_SOC_sizeNN as lka
    elif method.endswith("KoopmanNonlinearA"):
        import learn_DKAC_SOC_sizeNN as lka
    elif method.endswith("KoopmanU"):
        import learn_DKUC_SOC_sizeNN as lka
    elif method.endswith("KNonlinearmamba"):
        import Learn_Knonlinear_mamba as lka  
    for file in os.listdir(root_path):
        if file.startswith(method+"_"+"henon_map") and file.endswith(".pth"):
            model_path = file  
    # Data_collect = data_collecter(env_name)
    udim = 1
    Nstates = 2
    layer_depth = layer_i
    layer_width = 128
    dicts = torch.load(root_path+"/"+model_path,map_location=torch.device('cpu'))
    state_dict = dicts["model"]
    if method.endswith("KNonlinear"):
        Elayer = dicts["Elayer"]
        net = lka.Network(layers=Elayer,u_dim=0)
    elif method.endswith("KNonlinearRNN"):
        net = lka.Network(input_size=0+2,output_size=Nstates,hidden_dim=layer_width, n_layers=layer_depth-1)
    elif method.endswith("KNonlinearmamba"):
        net = LSPN_Mamba(
        # This module uses roughly 3 * expand * d_model^2 parameters
        d_model=2, # Model dimension d_model
        d_state=8,  # SSM state expansion factor
        d_conv=4,    # Local convolution width
        expand=4,    # Block expansion factor
    ).to("cuda")
    elif method.endswith("KoopmanNonlinear") or method.endswith("KoopmanNonlinearA"):
        layer = dicts["layer"]
        blayer = dicts["blayer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,blayer,NKoopman,udim)
    elif method.endswith("KoopmanU"):
        layer = dicts["layer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,NKoopman,udim)  
    net.load_state_dict(state_dict)
    #device = torch.device("cpu")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.cuda()
    net.double()
    Samples = 1
    steps = steps
    random.seed(2022)
    np.random.seed(2022)
    times = 100
    pre_list_all = np.zeros((times,steps-1,2))
    rea_list_all = np.zeros((times,steps-1,2))
    with torch.no_grad():
        for i in range(times):
            # test_data_path = "D:/毕业设计/中期/Python/MPC_trykoopman/results/SOC_compare_sizeNNdata/"+"method{}{}.npy".format(env_name,i)
            # if os.path.exists(test_data_path):
            #     test_data = np.load("D:/毕业设计/中期/Python/MPC_trykoopman/results/SOC_compare_sizeNNdata/{}{}.npy".format(env_name,i))
            # else:
            X_original_shape = read_henon_map_dataset_original_shape('/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/utility_LSPN/henon_map_data_filtered.npy')
            test_data = X_original_shape[-Samples:, :steps, :]
            np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_data/middle_data/heatdatapredict_future{}{}{}.npy".format(i,suffix,env_name),test_data)
            _,_,_,pre_list,rea_list = short_predict(method_index,test_data,net)
            #print(pre_list.shape)
            pre_list_all[i,:,:] = pre_list
            rea_list_all[i,:,:] = rea_list
    pre_list_mean = pre_list_all#np.mean(pre_list_all,axis=0)
    rea_list_mean = rea_list_all#np.mean(rea_list_all,axis=0)  
    # np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/"+env_name+"_"+method+"layer1{}{}.npy".format(layer_i, steps),np.array([pre_list_mean, rea_list_mean]))
    np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/heatdata"+env_name+"{}-pre-long{}.npy".format(method_index,steps),pre_list_mean)
    np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/heatdata"+env_name+"{}-real-long{}.npy".format(method_index,steps),rea_list_mean)
    return pre_list_mean, rea_list_mean

In [6]:
suffix = ["*","*","Knolinear_SOC_models","KRNN_SOC_models","DKUC_SOC_sizeNN","compare_DKAC_sizeNN_30","DKN_SOC_sizeNN","mamba_testhenon_map1"]
env_name = "henon_map"

steps = 101
for i in range(7,8):
    pre_list_mean, rea_list_mean = mean_predict(suffix[i],env_name,method_index=i,layer_i=2,steps=steps)
    print(pre_list_mean.shape)

KNonlinearmamba
tensor(-0.4044, device='cuda:0') tensor(-0.4060, device='cuda:0')
tensor(1.0306, device='cuda:0') tensor(0.9940, device='cuda:0')
tensor(-0.5142, device='cuda:0') tensor(-0.5051, device='cuda:0')
tensor(0.9316, device='cuda:0') tensor(0.9410, device='cuda:0')
tensor(-0.4071, device='cuda:0') tensor(-0.3911, device='cuda:0')
tensor(1.0682, device='cuda:0') tensor(1.0681, device='cuda:0')
tensor(-0.7227, device='cuda:0') tensor(-0.7146, device='cuda:0')
tensor(0.6018, device='cuda:0') tensor(0.6056, device='cuda:0')
tensor(0.2894, device='cuda:0') tensor(0.2722, device='cuda:0')
tensor(1.0704, device='cuda:0') tensor(1.0779, device='cuda:0')
tensor(-0.5517, device='cuda:0') tensor(-0.5451, device='cuda:0')
tensor(0.8997, device='cuda:0') tensor(0.9074, device='cuda:0')
tensor(-0.3295, device='cuda:0') tensor(-0.3163, device='cuda:0')
tensor(1.1358, device='cuda:0') tensor(1.1322, device='cuda:0')
tensor(-0.8887, device='cuda:0') tensor(-0.8894, device='cuda:0')
tensor(0.2